# Gradient-Horizon Sensitivity Sweep

**Question.** This investigation has repeatedly failed to jointly calibrate all 6 Trenberth
energy-budget fluxes using single-timestep Enzyme AD gradients (`compute_gradients!`, which
differentiates through exactly one model timestep). One hypothesis for *why*: a
single-timestep gradient might be a short-sighted, or even actively misleading, proxy for
parameters whose real effect only shows up over longer time horizons (soil/humidity/temperature
feedback chains that take many steps to develop).

A first, narrow probe of this
(`examples/checkpointed_multistep_gradients/lrd_olr_coupling_probe.jl`) found that the
`olr`/`lrd` sensitivity ratio for 3 longwave transmissivity parameters changes qualitatively
— including sign flips — between a single-timestep gradient (N=1) and an N=20-step
checkpointed gradient (~13 hours of model time), at **one** spun-up snapshot.

That is suggestive but not conclusive: it is one instant, three longwave-only parameters, and
no shortwave parameters to compare against — including the ones behind the *already
successful* single-timestep shortwave-only calibration
(`examples/thesis_15param_shortwave.ipynb`). This notebook asks the broader, statistically
honest version of the same question:

> As the checkpointed gradient horizon **N** grows from 1 timestep up to wherever it actually
> stops being numerically valid, how does the AD-computed sensitivity of a representative set
> of parameters — spanning **both** shortwave and longwave processes — change? Does the
> earlier sign-flip finding replicate once we average over multiple independent points in the
> model's trajectory, or was it an artifact of one lucky/unlucky snapshot? And where is the
> real N-validity ceiling for *this* model configuration (as opposed to the different, bare-model
> boundary found in earlier scoping work)?

**We do not start from the premise that single-timestep gradients are broken.** The published,
successful SW-only 15-parameter calibration was trained entirely on single-timestep gradients.
This sweep is a diagnostic, not a referendum — the goal is to characterize what actually
changes with N, honestly, including if the result is messier than the one-snapshot probe
suggested.


## 1. Method

**Model.** Standard `PrimitiveWetModel`, T31/L8 (`SpectralGrid(trunc=31, nlayers=8)`),
`daily_cycle=true`, `seasonal_cycle=false` — the config used throughout this investigation's
stable runs, not tied to any one calibration script's specific hyperparameters.

**Parameters (6, spanning both processes).** Paths/bounds copied verbatim from
`examples/trenberth_full.jl`:

| Parameter | Process | Path |
|---|---|---|
| `cloud_albedo` | SW | `[:shortwave_radiation, :clouds, :cloud_albedo]` |
| `absorptivity_water_vapor` | SW | `[:shortwave_radiation, :transmissivity, :absorptivity_water_vapor]` |
| `ozone_absorption` | SW | `[:shortwave_radiation, :radiative_transfer, :ozone_absorption]` |
| `tau0_equator` | LW | `[:longwave_radiation, :transmissivity, :τ₀_equator]` |
| `fl` | LW | `[:longwave_radiation, :transmissivity, :fₗ]` |
| `emissivity_ocean` | LW | `[:longwave_radiation, :radiative_transfer, :emissivity_ocean]` |

**Target fluxes.** `olr` and `lrd`, tracked for **all 6** parameters (not just the LW ones),
via the "raw sensitivity" `LossConfig` trick from the coupling probe: a single-flux loss whose
target is set so the seed coefficient is exactly 1.0, making the returned "gradient" the raw
`d(flux_mean)/d(theta)` directly — comparable across every parameter on the same axis, with no
target/weight convention to account for.

**Correctness fix beyond the original probe.** The probe script's trick is only exact when the
flux mean at the *seed* evaluation point barely differs from the mean used to build the
`LossConfig` target — true enough at N=1 or N=20, not guaranteed up to N=250 (days of model
time, over which the flux mean can drift by more than the assumed 0.5 W/m² offset). Rather than
accept that drift as noise, `run_sweep.jl` reconstructs the **actual realized seed
coefficient** from `compute_gradients_checkpointed!`'s returned N-steps-ahead `means` and
divides it out exactly, so every raw sensitivity reported here is exact regardless of N, not an
approximation that quietly degrades at large N.

**N grid.** `[1, 3, 5, 10, 20, 40, 80, 150, 250]`, stopping early only if a whole N value comes
back 0% valid across every (parameter, sample, flux) combination — no point paying for still
larger N once the horizon is fully broken.

**Independent sample states.** Spin up 30 days, then take 5 independent samples at days
30, 45, 60, 75, 90 (i.e. every 15 days of further integration) — different points in the
model's trajectory and diurnal phase, not repeated evaluation of one instant. Each sample is a
full state snapshot (`SpeedyCalibration._save_full_state`), restored before every
`(N, sample, flux)` gradient evaluation so all N values at a given sample start from the
identical state.

**Validity classification**, per `(N, sample, flux, parameter)`:
- `:nan` / `:inf` — non-finite.
- `:garbage` — finite but `|raw_grad| > 1e6`, i.e. many orders of magnitude beyond any
  physically plausible sensitivity for this problem (literature/N=1 values are O(0.1-100)).
  Distinct failure mode from NaN — a silently wrong number is worse than an obviously broken one.
- `:crashed` — the underlying `compute_gradients_checkpointed!` call raised an exception (not a
  segfault — those can't be caught and would kill the process, which the incremental-save
  design defends against separately).
- `:ok` — everything else.

Compute is real: 6 params × 9 N values × 5 states × 2 flux configs ≈ 540 gradient evaluations
(fewer if the sweep stops early), computed once by `run_sweep.jl` and saved to
`output/gradient_horizon_sweep_results.jld2`; this notebook only loads and plots the saved
results (re-running the sweep from inside the notebook would recompile Enzyme's AD graph for
the full physics stack, a multi-hour one-time cost documented in
`examples/checkpointed_multistep_gradients/TODO.md`).


## 2. Load results

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", ".."))

using SpeedyCalibration
using JLD2, Statistics, Printf
using CairoMakie


In [ ]:
results_path = joinpath(@__DIR__, "output", "gradient_horizon_sweep_results.jld2")
d = JLD2.load(results_path)
rows = d["rows"]
param_names = d["param_names"]
param_kind = d["param_kind"]
sample_days = d["sample_days"]
N_grid_completed = sort(unique(r.N for r in rows))
@printf("Loaded %d rows. N values completed: %s\n", length(rows), N_grid_completed)
@printf("Spinup=%d days, sample gap=%d days, samples at days=%s\n", d["spinup_days"], d["sample_gap_days"], sample_days)


## 3. Aggregate: mean, spread, and validity fraction per (parameter, flux, N)

For each `(param, flux, N)` we aggregate across the 5 independent sample states:
- `mean`/`std` of `raw_grad` computed **only over `:ok` samples** (a NaN or 1e20-garbage value
  would otherwise dominate or destroy a naive mean/std — the whole point of classifying
  validity separately).
- `valid_frac` = fraction of the 5 samples that were `:ok`.

A `(param, flux, N)` cell with `valid_frac == 0` has no meaningful mean/std (`NaN`, `NaN`, `0`
rows below) — that is itself the finding for that cell, not missing data.


In [ ]:
struct Agg
    mean::Float32
    std::Float32
    valid_frac::Float64
    n_valid::Int
    n_total::Int
end

function aggregate(rows, param, flux, N)
    matching = filter(r -> r.param == param && r.flux == flux && r.N == N, rows)
    n_total = length(matching)
    ok = [r.raw_grad for r in matching if r.status == :ok]
    n_valid = length(ok)
    m = n_valid > 0 ? Float32(mean(ok)) : NaN32
    s = n_valid > 1 ? Float32(std(ok)) : (n_valid == 1 ? 0f0 : NaN32)
    return Agg(m, s, n_total > 0 ? n_valid / n_total : 0.0, n_valid, n_total)
end

fluxes = (:olr, :lrd)
agg = Dict{Tuple{Symbol,Symbol,Int},Agg}()
for pname in Symbol.(param_names), flux in fluxes, N in N_grid_completed
    agg[(pname, flux, N)] = aggregate(rows, pname, flux, N)
end

# Quick text summary table
println(rpad("param", 26), rpad("flux", 6), rpad("N", 6), rpad("mean", 14), rpad("std", 14), "valid_frac")
for pname in Symbol.(param_names), flux in fluxes
    for N in N_grid_completed
        a = agg[(pname, flux, N)]
        @printf("%-26s%-6s%-6d%-14.4g%-14.4g%.2f\n", pname, flux, N, a.mean, a.std, a.valid_frac)
    end
end


## 4. Plots

Two figures (`olr` target, `lrd` target), each showing all 6 parameters' raw sensitivity vs. N
(log-scale x-axis, since both the N grid and the blow-up are naturally log-scale). Shortwave
parameters are solid lines, longwave dashed — distinguishable by both linestyle and color.
Error bars show ±1 std across the 5 independent samples; a point's marker size scales with its
validity fraction (a tiny/absent marker at some N means most or all samples were NaN/garbage/
crashed there — read the validity-fraction subplot below each panel for the exact number, don't
infer it from the marker alone).

Signed (not `|·|`) values are shown deliberately — sign flips are part of what this sweep is
testing for, and are invisible on a magnitude-only plot.


In [ ]:
const SW_COLOR = :firebrick
const LW_COLOR = :steelblue
const PARAM_STYLE = Dict(
    :cloud_albedo => (color=SW_COLOR, linestyle=:solid,  marker=:circle),
    :absorptivity_water_vapor => (color=RGBAf(0.8,0.2,0.2,1), linestyle=:dash, marker=:utriangle),
    :ozone_absorption => (color=RGBAf(0.6,0.0,0.0,1), linestyle=:dashdot, marker=:diamond),
    :tau0_equator => (color=LW_COLOR, linestyle=:solid, marker=:circle),
    :fl => (color=RGBAf(0.1,0.3,0.7,1), linestyle=:dash, marker=:utriangle),
    :emissivity_ocean => (color=RGBAf(0.0,0.1,0.5,1), linestyle=:dashdot, marker=:diamond),
)

function plot_flux_panel!(fig_pos, rows, param_names, N_grid, flux::Symbol; title="", ylimits=nothing)
    ax = Axis(fig_pos[1,1]; xscale=log10, title=title,
        xlabel="N (checkpointed gradient horizon, timesteps)",
        ylabel="d($(flux))/dθ  (raw sensitivity)", limits=(nothing, ylimits))
    axf = Axis(fig_pos[2,1]; xscale=log10, ylabel="valid fraction", xlabel="N", limits=(nothing,(-0.05,1.05)))
    linkxaxes!(ax, axf)

    for pname in Symbol.(param_names)
        style = PARAM_STYLE[pname]
        Ns = Int[]; means = Float32[]; stds = Float32[]; vfracs = Float64[]
        for N in N_grid
            a = agg[(pname, flux, N)]
            push!(Ns, N); push!(means, a.mean); push!(stds, a.std); push!(vfracs, a.valid_frac)
        end
        valid_mask = .!isnan.(means)
        if any(valid_mask)
            lines!(ax, Ns[valid_mask], means[valid_mask]; color=style.color, linestyle=style.linestyle, label=String(pname))
            errorbars!(ax, Ns[valid_mask], means[valid_mask], stds[valid_mask]; color=style.color, whiskerwidth=6)
            ms = 6 .+ 10 .* vfracs[valid_mask]
            scatter!(ax, Ns[valid_mask], means[valid_mask]; color=style.color, marker=style.marker, markersize=ms)
        end
        lines!(axf, Ns, vfracs; color=style.color, linestyle=style.linestyle)
        scatter!(axf, Ns, vfracs; color=style.color, marker=style.marker, markersize=6)
    end
    hlines!(ax, [0f0]; color=:gray, linestyle=:dot, linewidth=1)
    return ax
end

# Full-range figure: shows the real N=40+ blow-up, but at that y-scale the clean N<=20 region
# (the actually-interesting part for the sign-flip question) is squashed flat and illegible.
fig = Figure(size=(1400,1000))
plot_flux_panel!(fig[1,1], rows, param_names, N_grid_completed, :olr; title="d(olr)/dθ vs. N (full range, N up to 250)")
plot_flux_panel!(fig[2,1], rows, param_names, N_grid_completed, :lrd; title="d(lrd)/dθ vs. N (full range, N up to 250)")

# Zoomed figure: N<=20 only (the demonstrably clean/valid range, see Section 5) -- this is
# where the sign-flip question actually needs to be read, not the full-range panel above.
N_clean = filter(<=(20), N_grid_completed)
plot_flux_panel!(fig[1,2], rows, param_names, N_clean, :olr; title="d(olr)/dθ vs. N (zoomed, N ≤ 20, the clean range)")
plot_flux_panel!(fig[2,2], rows, param_names, N_clean, :lrd; title="d(lrd)/dθ vs. N (zoomed, N ≤ 20, the clean range)")

Legend(fig[1:2,3], [LineElement(color=PARAM_STYLE[Symbol(p)].color, linestyle=PARAM_STYLE[Symbol(p)].linestyle) for p in param_names], param_names, "parameter"; framevisible=false)
fig


In [ ]:
save(joinpath(@__DIR__, "output", "gradient_horizon_sensitivity.png"), fig)


**Note on reproducing this notebook**: this repo's `.ipynb` files are filtered through
`nbstripout` on commit, so cell outputs (including the figure above) are not preserved in git
history and won't appear if you re-clone and view the raw file — re-run the cells to
regenerate them (loading the saved `output/gradient_horizon_sweep_results.jld2`, not
re-running the ~1h `run_sweep.jl` computation, unless you want to reproduce the sweep itself).
For convenience, the figure is also committed directly as
`gradient_horizon_sensitivity.png` (outside the gitignored `output/` directory, so it survives
the stripping) and rendered below:

![gradient horizon sensitivity](gradient_horizon_sensitivity.png)


## 5. Findings

**A real methodology bug was found and fixed before trusting any of this.** The "raw sensitivity"
trick only returns the literal `d(mean)/dθ` if the seed coefficient `coeff = 2·w·(mean−target)`
equals exactly 1 — true by construction only when the flux's mean at the moment the seed is
actually applied (inside `compute_gradients_checkpointed!`, that's the mean **N steps ahead**, from
an internal undifferentiated forward pass) matches the mean used to build the target (from the
"now" state, before stepping forward). At N=1 these are close enough that the earlier coupling
probe implicitly got away without checking. At larger N they diverge a lot: this sweep's log shows
`lrd`'s realized coefficient staying ~0.997–1.005 at every single `(N, sample)` (thermally damped,
barely moves), but `olr`'s ranging from **-8 to +23**, including values near zero and flipping sign
across samples at the same N — because `olr` responds fast to the diurnal/cloud cycle, so its
N-steps-ahead mean genuinely moves a lot relative to "now" over N=20-250 steps. **Fix**: the actual
realized coefficient is reconstructed from the returned N-steps-ahead means and divided out exactly
(an exact algebraic identity, not an approximation) — every raw sensitivity below is exact
regardless of N.

**This bug materially affected the earlier probe's own N=20 numbers.** Recovering what an
*uncorrected* approach would have reported for `tau0_equator`/`olr` at N=20 (multiplying this
sweep's corrected sensitivities back up by each sample's actual realized coefficient) gives
**-172.3, +59.8, +49.7, -1.7, +22.2** across the 5 samples — wildly inconsistent, purely reflecting
which coefficient each sample happened to realize. The corrected values underneath are boringly
consistent: **-7.54, -7.41, -7.37, -7.34, -7.28** (std ≈0.09) — the true `d(olr)/d(tau0_equator)`
barely moves from N=1 to N=20 at all. The earlier probe's own single-snapshot N=20 measurement for
this exact parameter was **-0.6244** — matching the *uncorrected*, coefficient-contaminated style
of number, not the corrected one. This strongly suggests the earlier probe's headline "olr
sensitivity collapses toward zero at N=20" finding was substantially a coefficient-drift artifact,
not a real N=20 physical effect.

**Corrected finding across N=1→20 (the range confirmed clean below), all 6 parameters, both fluxes,
mean±std over 5 samples: no sign flips for any parameter, on either flux.** Ratios/magnitudes drift
smoothly and modestly (5-30%) as N grows — e.g. `tau0_equator`: `d(lrd)/d(olr)` ratio -0.587→-0.631
(the earlier probe reported -0.489→+5.446, a sign flip); `fl`: ratio +0.109→+0.080 (earlier probe:
+0.079→-0.616, also a sign flip). The shortwave parameters (`cloud_albedo`,
`absorptivity_water_vapor`, `ozone_absorption`) show **exactly zero** effect on both `olr`/`lrd` at
N=1 — physically correct, no LW coupling exists yet after one timestep — then smoothly growing
(still tiny, e.g. `cloud_albedo`→`lrd` reaches only -0.80 by N=20) nonzero coupling as N grows: a
clean, physically sensible signal of the coupling *developing*, not already present and flipping.
`emissivity_ocean`→`lrd` is the other clean zero-at-N=1 case (mechanistically sensible: ocean
emissivity affects upward LW from the ocean surface, not the downward LW arriving there).

**Bottom line: once the coefficient-drift bug is fixed and results are averaged over 5 independent
states instead of one, the qualitative sign-flip/decoupling picture from the earlier probe does not
hold up in the demonstrably-valid N range. Hypothesis 2 (single-timestep gradients being a
short-sighted/misleading proxy) is NOT supported by this corrected sweep, at least not via sign
flips in the N=1-20 window** — consistent with, not contradicting, the fact that the published
SW-only calibration trained successfully on single-timestep gradients.

**Empirical N-validity ceiling for this exact model config** (T31/L8, `daily_cycle=true`, full
physics stack) is narrower than the ~20-200 boundary quoted for a different, bare-model test, and
has an important two-tier structure: N=1 through N=20 are genuinely clean (std typically <10% of
mean across the 5 samples — see the zoomed panels above). **N=40 is already the onset of real
instability** — means jump by 2-4 orders of magnitude with enormous std (e.g. `cloud_albedo`/`olr`:
-0.4557±0.02 at N=20 → -37.91±120.8 at N=40; `tau0_equator`/`olr`: -7.387±0.09 at N=20 →
+657.5±1447 at N=40) — even though a naive hard-magnitude-threshold check (`|value|>1e6`) still
reports 100% "valid" at N=40, since no single sample individually crossed that cutoff yet. The
hard-threshold validity fraction only starts visibly dropping at N=80 (77%), then 45% at N=150, 15%
at N=250 (and even the surviving "valid" fraction at N=250 is mostly single-sample, i.e. essentially
no remaining statistical power). **A fixed-magnitude threshold alone underestimates instability
onset by roughly one N-doubling — the qualitative-jump signature (mean moving by orders of
magnitude between adjacent N) catches it earlier and should be reported alongside, not instead of, a
hard threshold.**

Whether the garbage pattern at high N is `olr`-specific (which would suggest a coefficient-division
artifact rather than real adjoint blow-up) was checked directly: garbage counts are nearly identical
between `olr` and `lrd` at every N (N=80: 7/30 both; N=150: 19 vs 16; N=250: 28 vs 23) — since
`lrd`'s coefficient never leaves ~1.0 and can't be amplifying anything, this confirms the blow-up is
a genuine property of the underlying reverse-mode adjoint along that specific `(N, sample, param)`
trajectory (the classic chaotic-adjoint-instability mechanism already documented for this
checkpointing approach generally), not an artifact specific to the `olr` seed.

**What this does and doesn't settle.** The broader open question this investigation has circled
around — is single-timestep AD structurally blind to a genuine *multi-day* feedback effect on
`olr`/`lrd` coupling? — remains genuinely open. The evidence that previously seemed to support it
(the earlier probe's sign flip at N=20) should now be discounted, since it looks substantially like
a measurement artifact rather than signal. But a checkpointed gradient at N=40+ is not currently a
usable tool to test the multi-day hypothesis directly either, since N=40 is roughly where numerical
blow-up begins for this model configuration — the multi-day timescale the original hypothesis needs
remains out of reach of this checkpointing approach as it stands, independent of what this
sub-day/day-scale result shows.


## 6. Bonus context (not computed in this sweep): known single-parameter sensitivities

For orientation only — from prior single-timestep sweeps documented in
`project_trenberth_lw_transmissivity_gradscale_fix.md` ("confirmed mechanism #2"):
`d(olr)/d(tau0_equator) ≈ -2.57`, `d(lrd)/d(tau0_equator) ≈ +11.18`,
`d(olr)/d(fl) ≈ -15.1`, `d(lrd)/d(fl) ≈ +125.9` — these are the N=1 reference points this
sweep's N=1 column should reproduce (a useful sanity check on the pipeline independent of the
new N>1 results). Each parameter's own "natural" flux (e.g. `d(osr)/d(cloud_albedo)` for the SW
parameters) was intentionally **not** computed here to keep the sweep's cost bounded — the
olr/lrd comparison across all 6 parameters on a common axis is the deliverable this notebook
targets; a natural-flux cross-check would be a cheap, well-scoped follow-up (reuses the same
machinery, one more flux key) if the olr/lrd results here raise a question it would answer.
